# 🔴 Solution: Sliding Window Attention（面试版）

## 📋 题目描述

**难度：** Medium

实现滑动窗口注意力。

滑动窗口注意力限制每个位置只关注固定窗口内的位置，在保持局部上下文的同时降低长序列的复杂度。

**签名:** `sliding_window_attention(Q, K, V, window_size) -> Tensor`

**参数:**
- `Q`, `K`, `V` — 输入张量 (B, S, D)
- `window_size` — 每个位置关注 |i-j| <= window_size 范围内的位置

**返回:** 注意力输出 (B, S, D)

**约束:**
- 用 `-inf` 掩盖 `|i - j| > window_size` 的位置
- `window_size=0` 表示每个位置只能看到自身
- 大窗口等同于全注意力

**提示：** 标准注意力 + 窗口遮蔽。构造 `|i-j|` 矩阵 → 将 `|i-j| > window_size` 的位置填 `-inf` → softmax → `@ V`。


In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW

def sliding_window_attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, window_size: int, mask: torch.Tensor = None) -> torch.Tensor:
    d_k = key.size(-1)
    seq_len = query.size(-2)

    # ---- Step 1: 注意力分数 ----
    scores = query @ key.transpose(-2, -1) / math.sqrt(d_k)

    # ---- Step 2: 滑动窗口 mask ----
    # 位置 i 和 j 的距离 |i-j| > window_size 则 mask
    positions = torch.arange(seq_len, device=query.device)
    # unsqueeze(0) 和 unsqueeze(1) 配合广播得到 [S, S] 距离矩阵
    rel_dist = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()
    window_mask = rel_dist > window_size

    # ---- Step 3: 应用 mask ----
    scores = scores.masked_fill(window_mask, float('-inf'))

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # ---- Step 4: softmax + 求和 ----
    scores_max = scores.max(dim=-1, keepdim=True).values
    exp_scores = torch.exp(scores - scores_max)
    attn_weights = exp_scores / exp_scores.sum(dim=-1, keepdim=True)

    return attn_weights @ value

# 面试追问：
# Q: 滑动窗口注意力的复杂度？
# A: O(n × w)，w 为窗口大小，相比全局 O(n²) 大幅降低
# Q: 如何捕获全局信息？
# A: 可以交替使用全局和局部注意力层（如 Longformer）

In [ ]:
Q=torch.randn(1,6,8); K=torch.randn(1,6,8); V=torch.randn(1,6,8)
print('window=0==V?', torch.allclose(sliding_window_attention(Q,K,V,0), V, atol=1e-5))

## 📝 核心思路总结

1. **局部注意力**：每个位置只关注窗口大小范围内的位置
2. **距离 mask**：|i-j| > window_size 的位置设为 -inf
3. **复杂度优化**：O(n²) → O(n×w)
4. **长序列处理**：Longformer、Mistral 等模型的基础